# Audit Risk Scoring — Methodology & Decision Log

This notebook assigns a composite audit risk score to every provider-DRG record in the cleaned CMS Medicare Inpatient dataset. The goal is to produce an explainable, financially grounded risk ranking that a compliance team can use to prioritise which providers to review first.

---

## 1. Scoring Philosophy

The model is a **rule-based, additive weighted score** — not a machine learning model. This is a deliberate choice:

- A compliance officer needs to be able to look at any provider's score and understand exactly why it was flagged
- An explainable score that can be defended in a regulatory context is more valuable than a black-box model with marginally better predictive accuracy
- The three components map directly to the analytical findings from the EDA and SQL analysis notebooks, so the scoring logic is grounded in observed data patterns rather than arbitrary features

---

## 2. Score Components

The composite score has three components, each scaled to a maximum number of points:

| Component | Max Points | Signal | Direction |
|---|---|---|---|
| `ratio_score` | 50 | Charge-to-payment ratio | Higher ratio → higher score |
| `volume_score` | 35 | Total Medicare discharges | Lower volume → higher score |
| `outlier_score` | 15 | IQR statistical outlier flag | Flagged → 15 points, else 0 |
| **Total** | **100** | | |

**Maximum possible score: 100**

---

## 3. Component Details & Rationale

### 3.1 Ratio Score (0–50 points)

The charge-to-payment ratio is the primary analytical signal in this dataset — it directly measures how much a hospital charges relative to what Medicare pays. It receives the highest weight (50 points) because it is the most meaningful and directly observable indicator of billing anomaly.

**Implementation:**
- The ratio is clipped at the 99th percentile before scaling. This prevents a single extreme outlier from compressing every other hospital's score toward zero.
- After clipping, the ratio is normalised to 0–1 by dividing by the 99th percentile cap, then scaled to 0–50.

**Formula:** `(min(ratio, ratio_99th_pct) / ratio_99th_pct) × 50`

### 3.2 Volume Score (0–35 points) — Inverted

Low-volume providers receive higher scores because small sample sizes are easier to manipulate and harder for CMS to detect through routine monitoring. A hospital billing Medicare for only 5 cases of a given DRG has much less scrutiny than one billing 500 cases.

**Implementation:**
- Discharge counts are log-transformed before scaling. Without the log transform, a handful of very large hospitals would stretch the scale so far that all small hospitals would compress into an indistinguishable cluster near the maximum score.
- `np.log1p(x)` is used (log of 1+x) to safely handle very low discharge counts without log(0) errors.
- The normalised log volume is **inverted** (1 minus the value) so that low volume maps to high score.

**Formula:** `(1 - (log(1+discharges) - min) / (max - min)) × 35`

### 3.3 Outlier Score (0 or 15 points) — Binary

Records flagged as statistical outliers by the IQR method receive 15 bonus points. This is a binary signal — a record either trips the threshold or it doesn't.

**Implementation:**
- IQR multiplier: 1.5 (standard threshold)
- Lower bound floored at 1.0 — a charge-to-payment ratio below 1.0 is logically impossible under Medicare IPPS rules (payment cannot exceed charge), so the statistical lower bound is overridden by domain logic
- 5.36% of records are flagged as outliers at this threshold

**Weight rationale:** The outlier flag was originally weighted at 30 points but this caused the binary flag to dominate the score distribution — records either jumped past the High tier directly into Critical (if flagged) or stayed in Low/Medium (if not flagged), with very few records landing naturally in High. Reducing the weight to 15 points makes the flag a supporting confirmation signal rather than a deciding factor, which produces a more analytically meaningful tier distribution.

---

## 4. Scoring Iteration Log

The final weights and bin edges were arrived at through iteration, not set arbitrarily. This log documents the key decisions made.

### Iteration 1 — Original weights (40/30/30), bins (0/40/65/85/100)

```
Low         89,434
Medium      48,978
Critical     6,501
High         1,338
```

**Problem:** Critical (6,501) was larger than High (1,338), which is logically wrong — records were jumping from Medium directly into Critical due to the 30-point binary outlier flag, skipping the High tier almost entirely. The bins 65–85 and 85–100 were essentially empty ranges because 90% of all scores fell below 49.8.

**Root cause:** Original bins were arbitrary round numbers, not derived from the actual score distribution. The outlier_score weight of 30 points was large enough to push flagged records past the High tier on its own.

**Score distribution (original weights):**
```
mean:   40.9
std:    13.4
25th:   34.3
50th:   38.2
75th:   43.1
90th:   49.8
95th:   82.0
```

### Iteration 2 — Revised weights (50/35/15), bins derived from percentiles

Weights were revised to reduce outlier_score dominance. Points redistributed to ratio_score and volume_score to maintain a maximum of 100.

**Score distribution (revised weights):**
```
mean:   47.8
std:    12.4
25th:   40.7
50th:   45.5
75th:   51.5
90th:   59.9
95th:   78.1
```

Bin edges were derived from actual score percentiles rather than round numbers:
- Low ceiling set at 43 (just above 25th percentile)
- Medium ceiling set at 50 (just above median)
- High ceiling set at 78 (95th percentile)
- Critical: top 5% of scores

**Final distribution:**
```
Low         53,583  (36.6%)
Medium      49,284  (33.7%)
High        36,056  (24.7%)
Critical     7,328   (5.0%)
```

**Why this distribution is correct:**
- No single tier dominates — the four tiers capture 37/34/25/5% of records respectively
- Critical and High are now clearly ordered (High > Critical in terms of score range, Critical is rarer than High)
- The progression Low → Medium → High → Critical is monotonic across all three component means
- Critical at 5% (~7,300 records) represents a realistic audit workload scope

### Component mean by tier (final model)

| Tier | ratio_score | volume_score | outlier_score |
|---|---|---|---|
| Low | 9.2 | 26.4 | 0.0 |
| Medium | 16.8 | 31.4 | 0.0 |
| High | 35.9 | 30.4 | 9.6 |
| Critical | 44.6 | 32.2 | 15.0 |

The clean progression across all three components confirms the scoring model is differentiating risk levels as intended.

---

## 5. Final Parameters

```python
# Weights
ratio_score:   0–50 points
volume_score:  0–35 points
outlier_score: 0–15 points (binary: 15 if flagged, 0 if not)
Maximum possible score: 100

# Outlier detection
IQR multiplier: 1.5
Lower bound floor: 1.0 (domain logic override)
Outlier rate: 5.36% of records flagged

# Risk tier bins (derived from score percentiles)
Low:      0–43    53,583 records  (36.6%)
Medium:  43–50    49,284 records  (33.7%)
High:    50–78    36,056 records  (24.7%)
Critical: 78–100   7,328 records   (5.0%)
Total:           146,251 records
```

---

## 6. Limitations

- **Weights are judgment-based:** The 50/35/15 split reflects analytical reasoning about which signals matter most, but has not been validated against known audit outcomes. If historical CMS audit findings were available, the weights could be calibrated empirically.
- **Volume score compresses the middle:** The log-transformed volume score pushes most records toward a similar range (mean ~30 points regardless of tier), which limits its discriminatory power. A better volume signal might use a percentile-based ranking within DRG category rather than raw discharge counts.
- **Binary outlier flag:** A continuous outlier score (e.g. how many standard deviations above the IQR bound a record sits) would be more informative than the current 0/15 binary. This is a candidate improvement for a v2 model.
- **No temporal dimension:** This analysis covers a single annual release. A provider whose ratio has been increasing year-over-year is more concerning than one with a stable high ratio. Trend analysis across multiple CMS annual releases would strengthen the model significantly.

In [4]:
# 04_audit_risk_scoring.ipynb
# Run once — sets working directory to project root regardless of where notebook is
import pandas as pd
import numpy as np
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

# ── Three-component risk score ───────────────────────────────
# Component 1: Ratio score (0-40 points) — most important driver
# Component 2: Volume score (0-30 points) — low volume = more suspicious
# Component 3: Outlier flag (0-30 points) — binary IQR flag from cleaning

df = pd.read_csv('data/processed/cms_clean.csv', dtype={'Rndrng_Prvdr_CCN': str})

# ── Component 1: normalise ratio to 0-40 scale ───────────────
ratio_max = df['charge_to_payment_ratio'].quantile(0.99)  # cap at 99th pct
df['ratio_score'] = (
    (df['charge_to_payment_ratio'].clip(upper=ratio_max) / ratio_max) * 50
).round(1)

# ── Component 2: inverse volume score (low volume = high score) ─
# Log-transform discharges to compress the scale, then invert
log_vol = np.log1p(df['Tot_Dschrgs'])
df['volume_score'] = (
    (1 - (log_vol - log_vol.min()) / (log_vol.max() - log_vol.min())) * 35
).round(1)

# ── Component 3: outlier flag score ──────────────────────────
df['outlier_score'] = df['ratio_outlier_flag'].astype(int) * 15

# ── Composite score ──────────────────────────────────────────
df['audit_risk_score'] = (
    df['ratio_score'] + df['volume_score'] + df['outlier_score']
).round(1)

# ── Risk tier ────────────────────────────────────────────────
df['risk_tier'] = pd.cut(
    df['audit_risk_score'],
    bins=[0, 43, 50, 78, 100],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print(df['risk_tier'].value_counts())

# Step 1: understand the score distribution
print(df['audit_risk_score'].describe().round(1))

# Step 2: check percentiles to inform bin edges
print(df['audit_risk_score'].quantile([0.25, 0.50, 0.75, 0.90, 0.95]).round(1))

# Step 3: check component averages to catch dominance issues
print(df.groupby('risk_tier')[['ratio_score', 'volume_score', 'outlier_score']].mean().round(1))
print('unique providers:', df[df['risk_tier'] == 'Critical']['Rndrng_Prvdr_CCN'].nunique())
#print(df['ratio_outlier_flag'].value_counts())
#print(df['ratio_outlier_flag'].mean() * 100, '% flagged')
df.to_csv('data/processed/cms_audit_scored.csv', index=False)


Working directory: c:\Users\gamer\Documents\projects\cms-medicare-audit-analysis
risk_tier
Low         53583
Medium      49284
High        36056
Critical     7328
Name: count, dtype: int64
count    146251.0
mean         47.8
std          12.4
min          11.2
25%          40.7
50%          45.5
75%          51.5
max         100.0
Name: audit_risk_score, dtype: float64
0.25    40.7
0.50    45.5
0.75    51.5
0.90    59.9
0.95    78.1
Name: audit_risk_score, dtype: float64
           ratio_score  volume_score  outlier_score
risk_tier                                          
Low               10.1          27.9            0.0
Medium            14.8          31.5            0.0
High              23.7          32.1            0.2
Critical          42.5          31.4           15.0
unique providers: 550
